---------------------------------
Exploratory Data Analysis (EDA) charts for the QML Benchmarking literature-review
dataset (QML_benchmarking_cleaned.xlsx, sheet "Papers").

Requirements:
    pip install pandas openpyxl matplotlib

Usage:
    python generate_visualizations.py
    -> writes PNG figures into ./figures/

All figures use only matplotlib (no seaborn) so they render identically anywhere,
and use a colour-blind-friendly qualitative palette throughout for consistency.

In [15]:
!pip install matplotlib pandas openpyxl

In [2]:
"""
generate_visualizations.py
---------------------------------
Exploratory Data Analysis (EDA) charts for the QML Benchmarking literature-review
dataset (QML_benchmarking_cleaned.xlsx, sheet "Papers").

Requirements:
    pip install pandas openpyxl matplotlib

Usage:
    python generate_visualizations.py
    -> writes PNG figures into ./figures/

All figures use only matplotlib (no seaborn) so they render identically anywhere,
and use a colour-blind-friendly qualitative palette throughout for consistency.
"""

import os
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm

# ------------------------------------------------------------------
# 0. Setup
# ------------------------------------------------------------------
DATA_PATH = "QML_benchmarking_cleaned.xlsx"
OUT_DIR = "figures"
os.makedirs(OUT_DIR, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 140,
    "font.size": 11,
    "font.family": "DejaVu Sans",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

PALETTE = ["#0072B2", "#D55E00", "#009E73", "#E69F00", "#CC79A7",
           "#56B4E9", "#F0E442", "#7F3C8D"]

# Distinct grey shades reserved for "no information" style categories, so grey
# still reads as "no info" family at a glance, but each shade of "no" stays
# visually distinguishable in legends / stacked bars (darkest = most negative).
NEUTRAL_PALETTE = ["#BFBFBF", "#8C8C8C", "#595959", "#333333"]
NEUTRAL_LABELS = {"not specified", "not reported", "no", "unclear", "other",
                   "no comparison made", "no / unclear"}


def ordered_colors(labels):
    """Assign PALETTE colors to `labels` in order, but map any label that
    matches NEUTRAL_LABELS (case-insensitive) to a distinct grey shade from
    NEUTRAL_PALETTE instead of a normal palette color, so 'no info' categories
    stay visually related (all greys) while still being distinguishable from
    each other. Cycles through each palette if there are more labels than
    available colors."""
    colors = []
    palette_cycle = itertools.cycle(PALETTE)
    neutral_cycle = itertools.cycle(NEUTRAL_PALETTE)
    for lab in labels:
        if str(lab).strip().lower() in NEUTRAL_LABELS:
            colors.append(next(neutral_cycle))
        else:
            colors.append(next(palette_cycle))
    return colors

df = pd.read_excel(DATA_PATH, sheet_name="Papers")


def save(fig, name):
    path = os.path.join(OUT_DIR, name)
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    print("saved", path)


def pie(series, title, filename, min_pct_for_label=3, colors=None):
    counts = series.value_counts()
    fig, ax = plt.subplots(figsize=(7, 6))
    colors = colors or ordered_colors(counts.index)
    wedges, texts, autotexts = ax.pie(
        counts.values,
        labels=None,
        autopct=lambda p: f"{p:.0f}%" if p >= min_pct_for_label else "",
        startangle=90,
        colors=colors[: len(counts)],
        pctdistance=0.75,
        wedgeprops=dict(width=0.55, edgecolor="white"),
    )
    ax.legend(
        wedges,
        [f"{idx} (n={v})" for idx, v in counts.items()],
        title=title,
        loc="center left",
        bbox_to_anchor=(1.0, 0.5),
        frameon=False,
    )
    ax.set_title(title, fontsize=13, fontweight="bold")
    save(fig, filename)


# ------------------------------------------------------------------
# 1. Pie chart — Data nature (Data_Type_Category)
# ------------------------------------------------------------------
pie(df["Data_Type_Category"], "Distribution of Data Modalities Used", "01_data_type_pie.png")

# ------------------------------------------------------------------
# 2. Papers per year — bar + cumulative line (classic "growth" chart)
# ------------------------------------------------------------------
year_counts = df["Year"].value_counts().sort_index()
fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.bar(year_counts.index.astype(str), year_counts.values, color=PALETTE[0], label="Papers per year")
ax1.set_ylabel("Number of papers", color=PALETTE[0])
ax1.set_xlabel("Year")
ax1.set_title("Number of QML Benchmarking Papers per Year", fontsize=13, fontweight="bold")
for x, y in zip(year_counts.index.astype(str), year_counts.values):
    ax1.text(x, y + 0.15, str(y), ha="center", fontsize=9)

ax2 = ax1.twinx()
cum = year_counts.cumsum()
ax2.plot(year_counts.index.astype(str), cum.values, color=PALETTE[1], marker="o", linewidth=2, label="Cumulative")
ax2.set_ylabel("Cumulative number of papers", color=PALETTE[1])
fig.tight_layout()
save(fig, "02_papers_per_year.png")

# ------------------------------------------------------------------
# 3. Contribution type — new method proposal vs benchmark-only vs both
# ------------------------------------------------------------------
counts = df["Contribution_Type"].value_counts()
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(counts.index, counts.values, color=ordered_colors(counts.index))
ax.invert_yaxis()
ax.set_xlabel("Number of papers")
ax.set_title("Type of Contribution: Proposed Methods vs. Benchmark Studies", fontsize=12, fontweight="bold")
for b in bars:
    ax.text(b.get_width() + 0.3, b.get_y() + b.get_height() / 2, int(b.get_width()), va="center")
save(fig, "03_contribution_type.png")

# ------------------------------------------------------------------
# 4. Real vs synthetic datasets
# ------------------------------------------------------------------
pie(df["Dataset_Nature"], "Dataset Nature: Real vs. Synthetic", "04_dataset_nature_pie.png")

# ------------------------------------------------------------------
# 5. Real-world application or not
# ------------------------------------------------------------------
pie(df["Real_World_Application"], "Targets a Real-World Application?", "05_real_world_application_pie.png")

# ------------------------------------------------------------------
# 6. Library used (primary library)
# ------------------------------------------------------------------
pie(df["Primary_Library"], "Primary Software Library / Framework Used", "06_primary_library_pie.png")

# 6b. Bonus: since many papers use several libraries, also show co-occurrence
lib_cols = [c for c in df.columns if c.startswith("Uses_") and c not in ("Uses_Multiple_Libraries",)]
lib_usage = (df[lib_cols] == "Yes").sum().sort_values(ascending=False)
lib_usage.index = [c.replace("Uses_", "") for c in lib_usage.index]
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(lib_usage.index, lib_usage.values, color=PALETTE[1])
ax.set_ylabel("Number of papers mentioning the library")
ax.set_title("Software Library Usage (papers can use more than one)", fontsize=12, fontweight="bold")
plt.xticks(rotation=40, ha="right")
for b in bars:
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.3, int(b.get_height()), ha="center", fontsize=9)
save(fig, "06b_library_usage_bar.png")

# ------------------------------------------------------------------
# 7. Simulator vs real quantum computer
# ------------------------------------------------------------------
pie(df["Hardware_Type"], "Simulator vs. Real Quantum Hardware", "07_hardware_type_pie.png")

# ------------------------------------------------------------------
# 8. Quantum advantage reported (overall)
# ------------------------------------------------------------------
pie(df["Quantum_Advantage_Reported"], "Reported Quantum Advantage over Classical Baselines",
    "08_quantum_advantage_pie.png")

# ------------------------------------------------------------------
# 9. Task category distribution
# ------------------------------------------------------------------
counts = df["Task_Category"].value_counts()
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(counts.index, counts.values, color=PALETTE[2])
ax.invert_yaxis()
ax.set_xlabel("Number of papers")
ax.set_title("Machine Learning Task Addressed", fontsize=12, fontweight="bold")
for b in bars:
    ax.text(b.get_width() + 0.3, b.get_y() + b.get_height() / 2, int(b.get_width()), va="center")
save(fig, "09_task_category_bar.png")

# ------------------------------------------------------------------
# 10. Data type x Task heatmap (combine two columns)
# ------------------------------------------------------------------
ct = pd.crosstab(df["Data_Type_Category"], df["Task_Category"])
task_order = ct.sum(axis=0).sort_values(ascending=False).index
ct = ct[task_order]
fig, ax = plt.subplots(figsize=(10, 5.5))
im = ax.imshow(ct.values, cmap="Blues")
ax.set_xticks(range(len(ct.columns)))
ax.set_xticklabels(ct.columns, rotation=45, ha="right")
ax.set_yticks(range(len(ct.index)))
ax.set_yticklabels(ct.index)
for i in range(ct.shape[0]):
    for j in range(ct.shape[1]):
        v = ct.values[i, j]
        if v:
            ax.text(j, i, v, ha="center", va="center",
                     color="white" if v > ct.values.max() / 2 else "black", fontsize=9)
ax.set_title("Data Modality vs. ML Task (paper counts)", fontsize=12, fontweight="bold")
fig.colorbar(im, ax=ax, shrink=0.8, label="Number of papers")
fig.tight_layout()
save(fig, "10_datatype_vs_task_heatmap.png")

# ------------------------------------------------------------------
# 11. Quantum advantage vs Hardware type (stacked bar, %)
# ------------------------------------------------------------------
ct = pd.crosstab(df["Hardware_Type"], df["Quantum_Advantage_Reported"], normalize="index") * 100
fig, ax = plt.subplots(figsize=(9, 5.5))
bottom = np.zeros(len(ct))
seg_colors = ordered_colors(ct.columns)
for i, col in enumerate(ct.columns):
    ax.bar(ct.index, ct[col].values, bottom=bottom, label=col, color=seg_colors[i])
    bottom += ct[col].values
ax.set_ylabel("% of papers")
ax.set_title("Reported Quantum Advantage by Hardware Type", fontsize=12, fontweight="bold")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False, fontsize=9)
plt.xticks(rotation=15, ha="right")
fig.tight_layout()
save(fig, "11_advantage_by_hardware_stacked.png")

# ------------------------------------------------------------------
# 12. Papers per year, stacked by data modality (trend over time)
# ------------------------------------------------------------------
ct = pd.crosstab(df["Year"], df["Data_Type_Category"])
ct = ct.reindex(sorted(ct.index))
fig, ax = plt.subplots(figsize=(9, 5.5))
bottom = np.zeros(len(ct))
seg_colors = ordered_colors(ct.columns)
for i, col in enumerate(ct.columns):
    ax.bar(ct.index.astype(str), ct[col].values, bottom=bottom, label=col, color=seg_colors[i])
    bottom += ct[col].values
ax.set_ylabel("Number of papers")
ax.set_xlabel("Year")
ax.set_title("Publication Trend by Data Modality", fontsize=12, fontweight="bold")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False, fontsize=9)
fig.tight_layout()
save(fig, "12_trend_by_modality_stacked.png")

# ------------------------------------------------------------------
# 13. Has public code available (open-science indicator)
# ------------------------------------------------------------------
pie(df["Has_Public_Code"], "Public Code Repository Available", "13_has_code_pie.png")


# ------------------------------------------------------------------
# 14. Common critique / limitation themes (keyword-mined from "Critic")
# ------------------------------------------------------------------
CRITIQUE_THEMES = {
    "Noise / decoherence": r"noise|decoheren|error rate",
    "Small dataset / subset": r"small dataset|subset|only \d|limited data|130 sentences|525 points",
    "Scalability limits": r"scalab|qubit limit|qubit count|high-dimensional|cannot handle",
    "No / weak classical baseline": r"no compar|not compared|no classical|without classical",
    "Fixed / simple circuits": r"fixed circuit|circuits are fixed|generic approach|simple",
    "Limited hardware access": r"real hardware|physical hardware|nisq|hardware noise",
    "Reporting issues (train vs test, etc.)": r"training accuracy|not test accuracy|architetcure is not shown|not shown",
    "Future work / early-stage": r"future work|early exploratory|promising but limited",
}

critic_text = df["Critic"].dropna().str.lower()
theme_counts = {
    theme: critic_text.str.contains(pattern, regex=True, na=False).sum()
    for theme, pattern in CRITIQUE_THEMES.items()
}
theme_counts = pd.Series(theme_counts).sort_values(ascending=True)
theme_counts = theme_counts[theme_counts > 0]

fig, ax = plt.subplots(figsize=(9, 5.5))
bars = ax.barh(theme_counts.index, theme_counts.values, color=PALETTE[4])
ax.set_xlabel(f"Number of papers (out of {critic_text.shape[0]} with a recorded critique)")
ax.set_title("Most Common Reported Limitations / Critiques", fontsize=12, fontweight="bold")
for b in bars:
    ax.text(b.get_width() + 0.1, b.get_y() + b.get_height() / 2, int(b.get_width()), va="center", fontsize=9)
fig.tight_layout()
save(fig, "14_critique_themes_bar.png")

# ------------------------------------------------------------------
# 15. Evaluation metrics used (keyword-mined from "metrics")
# ------------------------------------------------------------------
METRIC_FAMILIES = {
    "Accuracy": r"accuracy",
    "Precision / Recall": r"precision|recall",
    "F1-score": r"f1",
    "AUC / ROC": r"auc|roc",
    "Loss": r"\bloss\b",
    "MSE / RMSE": r"\bmse\b|\brmse\b",
    "MAE / MAPE": r"\bmae\b|\bmape\b",
    "R\u00b2": r"r2|r\u00b2",
    "Execution / training time": r"time|runtime",
}
metrics_text = df["metrics"].dropna().str.lower()
metric_counts = {
    fam: metrics_text.str.contains(pat, regex=True, na=False).sum()
    for fam, pat in METRIC_FAMILIES.items()
}
metric_counts = pd.Series(metric_counts).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5.5))
bars = ax.barh(metric_counts.index, metric_counts.values, color=PALETTE[2])
ax.set_xlabel(f"Number of papers (out of {metrics_text.shape[0]} with recorded metrics)")
ax.set_title("Evaluation Metrics Used Across Studies", fontsize=12, fontweight="bold")
for b in bars:
    ax.text(b.get_width() + 0.1, b.get_y() + b.get_height() / 2, int(b.get_width()), va="center", fontsize=9)
fig.tight_layout()
save(fig, "15_metrics_used_bar.png")

# ------------------------------------------------------------------
# 16. Contribution type vs. quantum advantage (publication-bias check)
#     Do "new method proposal" papers report advantage more often than
#     pure benchmark studies that have no stake in a specific new model?
# ------------------------------------------------------------------
ct = pd.crosstab(df["Contribution_Type"], df["Quantum_Advantage_Reported"], normalize="index") * 100
ct = ct.reindex(ct.sum(axis=1).sort_values(ascending=False).index)
fig, ax = plt.subplots(figsize=(9.5, 5.5))
bottom = np.zeros(len(ct))
seg_colors = ordered_colors(ct.columns)
for i, col in enumerate(ct.columns):
    ax.bar(ct.index, ct[col].values, bottom=bottom, label=col, color=seg_colors[i])
    bottom += ct[col].values
ax.set_ylabel("% of papers")
ax.set_title("Reported Quantum Advantage by Type of Contribution\n(possible publication-framing bias check)",
              fontsize=12, fontweight="bold")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False, fontsize=9)
plt.xticks(rotation=20, ha="right")
fig.tight_layout()
save(fig, "16_advantage_by_contribution_type_stacked.png")

# ------------------------------------------------------------------
# 17. Dataset nature (real vs synthetic) vs. quantum advantage
# ------------------------------------------------------------------
ct = pd.crosstab(df["Dataset_Nature"], df["Quantum_Advantage_Reported"], normalize="index") * 100
fig, ax = plt.subplots(figsize=(9, 5.5))
bottom = np.zeros(len(ct))
seg_colors = ordered_colors(ct.columns)
for i, col in enumerate(ct.columns):
    ax.bar(ct.index, ct[col].values, bottom=bottom, label=col, color=seg_colors[i])
    bottom += ct[col].values
ax.set_ylabel("% of papers")
ax.set_title("Reported Quantum Advantage by Dataset Nature", fontsize=12, fontweight="bold")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False, fontsize=9)
plt.xticks(rotation=15, ha="right")
fig.tight_layout()
save(fig, "17_advantage_by_dataset_nature_stacked.png")

# ------------------------------------------------------------------
# 18. Benchmarking rigor over time (avg. # methods benchmarked / year)
# ------------------------------------------------------------------
rigor = df.groupby("Year")["Num_Methods_Benchmarked"].mean()
rigor = rigor.reindex(sorted(rigor.index))
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(rigor.index.astype(str), rigor.values, marker="o", linewidth=2, color=PALETTE[6])
ax.fill_between(range(len(rigor)), rigor.values, alpha=0.15, color=PALETTE[6])
ax.set_ylabel("Avg. number of methods benchmarked")
ax.set_xlabel("Year")
ax.set_title("Benchmarking Rigor Over Time", fontsize=12, fontweight="bold")
for x, y in zip(rigor.index.astype(str), rigor.values):
    ax.text(x, y + 0.1, f"{y:.1f}", ha="center", fontsize=9)
fig.tight_layout()
save(fig, "18_rigor_over_time_line.png")

print("\nAll figures written to:", os.path.abspath(OUT_DIR))

saved figures\01_data_type_pie.png
saved figures\02_papers_per_year.png
saved figures\03_contribution_type.png
saved figures\04_dataset_nature_pie.png
saved figures\05_real_world_application_pie.png
saved figures\06_primary_library_pie.png
saved figures\06b_library_usage_bar.png
saved figures\07_hardware_type_pie.png
saved figures\08_quantum_advantage_pie.png
saved figures\09_task_category_bar.png
saved figures\10_datatype_vs_task_heatmap.png
saved figures\11_advantage_by_hardware_stacked.png
saved figures\12_trend_by_modality_stacked.png
saved figures\13_has_code_pie.png
saved figures\14_critique_themes_bar.png
saved figures\15_metrics_used_bar.png
saved figures\16_advantage_by_contribution_type_stacked.png
saved figures\17_advantage_by_dataset_nature_stacked.png
saved figures\18_rigor_over_time_line.png

All figures written to: d:\QML benchmarking\Visualizations_QML_survey\figures
